<a href="https://colab.research.google.com/github/jiwonleelee/Deepfake-Detection-v2/blob/main/notebooks/02_baseline/baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from tqdm import tqdm
import pandas as pd
from google.colab import drive
import os
from collections import defaultdict
import gc
from torchvision.transforms import functional as F

In [ ]:
# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

In [ ]:
import sys
# src 폴더를 인식할 수 있도록 경로 추가
sys.path.append('/content/drive/MyDrive/Deepfake-Detection-v2')

from src.utils import set_seed, log_to_csv
from src.dataset import DeepfakeDataset, seed_worker
from src.trainer import train_one_epoch, validate_video_level, save_checkpoint, EarlyStopping, load_checkpoint

In [ ]:
# 로컬에서 압축 해제 (네트워크 지연 없음)
!unzip -q /content/drive/MyDrive/Deepfake-Detection-v2/Datasets/FF++/FF++.zip -d /content/FF++_Extract

In [ ]:
# 2. 체크포인트 저장 경로 설정 (본인의 드라이브 경로에 맞게 수정 가능)
CHECKPOINT_DIR = '/content/drive/MyDrive/Deepfake-Detection-v2/Checkpoints/01_baseline'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 3. 데이터셋 경로
CSV_PATH = '/content/drive/MyDrive/Deepfake-Detection-v2/Datasets/FF++/FF++_meta.csv'

print(f"✅ 체크포인트 저장 경로: {CHECKPOINT_DIR}")

In [ ]:
# 시드 설정
EXPERIMENT_SEED = 42
set_seed(EXPERIMENT_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 시드 {EXPERIMENT_SEED} 고정 완료 및 {device} 준비")

In [ ]:
class RandomCenterResizedCrop:
    """중심을 고정한 채 마진(Scale)만 랜덤하게 조절하여 리사이즈"""
    def __init__(self, size, min_ratio=0.52, max_ratio=0.60):
        self.size = size
        self.min_ratio = min_ratio
        self.max_ratio = max_ratio

    def __call__(self, img):
        # 512px 이미지의 52% ~ 60% 사이를 랜덤하게 선택
        ratio = random.uniform(self.min_ratio, self.max_ratio)
        w, h = img.size
        new_w, new_h = int(w * ratio), int(h * ratio)

        # 중심 크롭 후 리사이즈
        img = F.center_crop(img, (new_h, new_w))
        img = F.resize(img, (self.size, self.size))
        return img


In [ ]:
g = torch.Generator()
g.manual_seed(EXPERIMENT_SEED)

IMG_SIZE = 224

# 주피터 노트북에서 적용할 transform
train_transform = transforms.Compose([
    transforms.RandomRotation(15), # 회전을 먼저 해서 배경 여유분 활용
    RandomCenterResizedCrop(224, min_ratio=0.52, max_ratio=0.60), # 1.04~1.2 마진 효과
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 2.0))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 검증 시: 마진 12.5% (배율 1.125) 고정 센터 크롭
val_transform = transforms.Compose([
    # 512px 이미지의 중심에서 56.25% 영역을 잘라내면 마진 12.5% 상태가 됩니다.
    transforms.CenterCrop(int(512 * 0.5625)),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# 실험 대상 모델
model_names = ['convnext_tiny']

# 하이퍼파라미터
MAX_EPOCHS = 20  # 얼리스탑이 있으므로 넉넉히 잡습니다.
BATCH_SIZE = 32
LR = 1e-4

# 결과를 저장할 리스트
results_summary = []

# 데이터 로더 준비
train_dataset = DeepfakeDataset(CSV_PATH, split='train', transform=train_transform)
val_dataset = DeepfakeDataset(CSV_PATH, split='val', transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, worker_init_fn=seed_worker, generator=g)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
# 결과 저장용 리스트
results_summary = []

for m_name in model_names:
    print(f"\n" + "="*50)
    print(f"🚀 Experiment Target: {m_name}")
    print("="*50)

    # 1. 시드 고정 (재현성 확보)
    set_seed(EXPERIMENT_SEED)

    # 2. 모델 및 기본 객체 초기화
    model = timm.create_model(m_name, pretrained=True, num_classes=1).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    # 얼리스탑 객체 생성 (기본값으로 시작)
    early_stopping = EarlyStopping(patience=5, verbose=True)

    # 3. [핵심 수정] 체크포인트 로드 (학습 재개 여부 판단의 근거)
    start_epoch, best_acc, es_state = load_checkpoint(m_name, model, optimizer, CHECKPOINT_DIR, device)

    if es_state is not None:
        # 1. 저장된 상태 전체 복구 (counter, early_stop 등)
        early_stopping.__dict__.update(es_state)

        # 2. [핵심] 만약 best_score가 비어있다면 불러온 best_acc로 채워줌
        if early_stopping.best_score is None:
            early_stopping.best_score = best_acc

        print(f"🔄 {m_name} 상태 복구: Epoch {start_epoch}, Best Acc {best_acc:.4f}, Counter {early_stopping.counter}")

    # 5. [핵심 수정] 학습 완료 여부 판별 로직
    # 조건 A: 이미 얼리스탑이 발생했는가?
    # 조건 B: 이미 설정한 최대 에포크(MAX_EPOCHS)까지 완주했는가?
    if early_stopping.early_stop or start_epoch >= MAX_EPOCHS:
        reason = "Early Stopping 완료" if early_stopping.early_stop else "최대 에포크 완주"
        print(f"✅ {m_name}은(는) 이미 학습이 종료된 상태입니다. ({reason})")
        results_summary.append({'Model': m_name, 'Best Video Acc': best_acc})

        # 메모리 정리 후 다음 모델로 스킵
        del model, optimizer
        torch.cuda.empty_cache()
        gc.collect()
        continue

    # 6. 학습 루프 (start_epoch부터 재개)
    print(f"▶️ {m_name} 학습을 {start_epoch+1} 에포크부터 재개합니다.")

    for epoch in range(start_epoch, MAX_EPOCHS):
        print(f"\n[Epoch {epoch+1}/{MAX_EPOCHS}]")

        # 학습 및 검증
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, video_acc = validate_video_level(model, val_loader, criterion, device)

        print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"   Val Loss:   {val_loss:.4f} | Video-level Val Acc: {video_acc:.4f}")

        # 에포크별 로그 기록
        log_to_csv(m_name, epoch+1, train_loss, train_acc, val_loss, video_acc, f"{CHECKPOINT_DIR}/{m_name}_history.csv")

        # 베스트 모델 갱신 확인
        is_best = video_acc > best_acc
        if is_best:
            best_acc = video_acc
            print(f"✨ Best Model Updated! (Acc: {best_acc:.4f})")

        # 얼리스탑 판정 (카운터 업데이트)
        early_stopping(video_acc)

        # 체크포인트 저장 (갱신된 es_state 포함)
        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_acc': best_acc,
            'es_state': early_stopping.__dict__,  # 갱신된 얼리스탑 상태 저장
        }, is_best, m_name, CHECKPOINT_DIR)

        if early_stopping.early_stop:
            print(f"🛑 {m_name} Early Stopping triggered at epoch {epoch+1}!")
            break

    results_summary.append({'Model': m_name, 'Best Video Acc': best_acc})

    # 메모리 정리
    del model, optimizer
    torch.cuda.empty_cache()
    gc.collect()